In [0]:
%python
dbutils.widgets.text("catalog","default")
catalog=dbutils.widgets.get("catalog")

In [0]:
create table if not exists IDENTIFIER(:catalog || '.silver.stores') (
  store_id      string,
  store_name    string,
  city          string,
  state         string,
  region        string,
  opening_date  date
)
using delta;

merge into IDENTIFIER(:catalog || '.silver.stores') t
using (
  select 
    store_id,
    trim(store_name) as store_name,
    initcap(trim(city)) as city,
    trim(state) as state,
    trim(region) as region,
    cast(opening_date as date) as opening_date
 from IDENTIFIER(:catalog || '.bronze.stores')) s
on t.store_id = s.store_id
when matched then update set *
when not matched then insert *

In [0]:
create table if not exists IDENTIFIER(:catalog || '.silver.products') (
  product_id  string,
  name string,
  brand string,
  category_l1 string,
  category_l2 string,
  pack_size int,
  pack_unit string,
  mrp decimal(10,2),
  selling_price decimal(10,2),
  is_active boolean,
  tags array<string>,
  last_changed_snapshot_date date
)
using delta
TBLPROPERTIES ('delta.enableChangeDataFeed' = true);

merge into IDENTIFIER(:catalog || '.silver.products') t
using (
  select 
    product_id, product_name as name, brand, category.l1 as category_l1, category.l2 as category_l2, pack.size as pack_size, pack.unit as pack_unit, mrp, selling_price, is_active, tags, cast(snapshot_date as date) as last_changed_snapshot_date
    from IDENTIFIER(:catalog || '.bronze.products')) s
on t.product_id = s.product_id
when matched and (
  t.name is distinct from s.name
  or t.brand is distinct from s.brand
  or t.category_l1 is distinct from s.category_l1
  or t.category_l2 is distinct from s.category_l2
  or t.pack_size is distinct from s.pack_size
  or t.pack_unit is distinct from s.pack_unit
  or t.mrp is distinct from s.mrp
  or t.selling_price is distinct from s.selling_price
  or t.is_active is distinct from s.is_active
  or t.tags is distinct from s.tags
) then update set *
when not matched then insert *

In [0]:
create table if not exists IDENTIFIER(:catalog || '.silver.customers') (
  customer_id string,
  full_name string,
  email string,
  phone string,
  city string,
  tier string,
  signup_date date,
  effective_from date,
  effective_to date,
  is_current boolean
)
using delta
TBLPROPERTIES ('delta.enableChangeDataFeed' = true);

merge into IDENTIFIER(:catalog || '.silver.customers') t
using (
  select
    customer_id,                
    trim(full_name) as full_name,    
    lower(trim(email)) as email,                      
    right(regexp_replace(phone, '[^0-9]', ''), 10) as phone, 
    initcap(trim(city)) as city,            
    upper(trim(tier)) as tier,       
    coalesce(
      try_to_date(signup_date, 'yyyy-MM-dd'),
      try_to_date(signup_date, 'dd/MM/yyyy')
    ) as signup_date,            
    date '2026-03-01' as effective_from,         
    cast(null as date) as effective_to,              
    true as is_current             
  from IDENTIFIER(:catalog || '.bronze.customers')
  -- deduplicate: identical rows count once (one version per customer)
  qualify row_number() over (partition by customer_id order by _ingested_at desc) = 1
) s
on t.customer_id = s.customer_id
  and t.effective_from = s.effective_from
when matched and (
  t.full_name is distinct from s.full_name
  or t.email is distinct from s.email
  or t.phone is distinct from s.phone
  or t.city is distinct from s.city
  or t.tier is distinct from s.tier
  or t.signup_date is distinct from s.signup_date
  or t.effective_to is distinct from s.effective_to
  or t.is_current is distinct from s.is_current
) then update set *
when not matched then insert *
    